# 30 Queries: the network and the gate in plain SQL over the Delta tables

In [ ]:
# Who can reach a compliance or licensing contact in the Gulf? Two hops: contributor -> source -> topics, then who those sources could introduce.
display(spark.sql('''
SELECT c.name AS contributor, s.token, s.descriptor, s.region_norm AS region, s.sector, e.date, e.project, s.n_mentioned AS could_introduce
FROM gold_edges e
JOIN gold_contributors c ON c.entity_id = e.src
JOIN gold_sources s ON s.entity_id = e.dst
WHERE e.type = 'INTERVIEWED' AND (lower(s.topics) LIKE '%compliance%' OR lower(s.topics) LIKE '%licens%') AND s.region_norm = 'Gulf'
ORDER BY e.date DESC'''))

In [ ]:
# The gate: how many documents a person actually reads.
display(spark.sql("SELECT tier, approval_mode, count(*) AS n, round(avg(risk_score),3) AS mean_risk FROM silver_redacted GROUP BY tier, approval_mode ORDER BY n DESC"))

In [ ]:
# Over time: sources reached more than once, and second-degree reach through the people they named.
display(spark.sql("SELECT token, descriptor, contributors, n_transcripts, first_seen, last_seen, n_mentioned FROM gold_sources WHERE n_transcripts > 1 ORDER BY n_transcripts DESC"))
display(spark.sql("SELECT p.token, p.descriptor, p.relationship, s.token AS via_source, s.descriptor AS via_descriptor FROM gold_people p JOIN gold_sources s ON s.entity_id = split(p.via, '; ')[0] ORDER BY p.n_docs DESC LIMIT 20"))